# Feature 4: Behavioral Financial Advisor (RAG + LLM)
This feature provides a highly personalized, AI-driven financial advisor for freelancers. It combines **Retrieval-Augmented Generation (RAG)** with real user financial data (schema-aligned) to provide actionable, grounded advice.

**Core Capabilities:**
1. **RAG Knowledge Base**: Uses ChromaDB to retrieve curated financial knowledge (e.g., the 50/30/20 rule, installment guidelines).
2. **Data-Grounded Context**: Injects the user's real income, expenses, debts, and savings into the LLM prompt.
3. **Specialized Tools**: Provides discrete functions for affordability checks, installment impact analysis, and savings planning.


In [1]:
# Fix protobuf version conflict
import subprocess
import sys

try:
    import protobuf
    print(f"Current protobuf version: {protobuf.__version__}")
except:
    pass

print("Fixing protobuf version compatibility...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", "-q", "protobuf>=6.31.1"])
print("✅ Protobuf updated successfully")

Fixing protobuf version compatibility...
✅ Protobuf updated successfully


In [2]:
import os
import sys
import warnings
import pandas as pd
import numpy as np
from dotenv import load_dotenv

load_dotenv(override=True) 

# Suppress warnings for cleaner output
os.environ["TF_USE_LEGACY_KERAS"] = "1"
os.environ["TRANSFORMERS_NO_TF"] = "1"
warnings.filterwarnings("ignore")

# Make sure we can import from the project
sys.path.insert(0, os.path.abspath(''))

# We will need the Gemini API key for the chat functionality
if not os.environ.get("GEMINI_API_KEY"):
    print("⚠️ WARNING: GEMINI_API_KEY is not set. The chat functionality will not work without it.")
    print("You can set it via: os.environ['GEMINI_API_KEY'] = 'your-key'")
else:
    print("✅ GEMINI_API_KEY is set.")


✅ GEMINI_API_KEY is set.


## 1. Schema-Aligned Data Gathering
The advisor needs to understand the user's real financial situation. We load their profile, income, expenses, debts, savings, and account balances to build a comprehensive context.


In [3]:
from utils.data_loader import (
    get_user_profile, 
    get_income_stats, 
    get_expense_stats,
    get_debt_info, 
    get_savings_info, 
    get_account_summary
)
from features.income_forecast import forecast_income
from features.cashflow_smoothing import get_budget_plan
from features.debt_prioritizer import prioritize_debts

def gather_user_context(user_id: str):
    """Gather all schema-aligned financial data for a user."""
    print(f"Gathering data for {user_id}...")
    
    profile = get_user_profile(user_id)
    forecast = forecast_income(user_id, months_ahead=3)
    budget = get_budget_plan(user_id)
    debts_info = prioritize_debts(user_id, strategy="hybrid")
    acc_sum = get_account_summary(user_id)
    debt_det = get_debt_info(user_id)
    sav_det = get_savings_info(user_id)
    
    print("✅ Data gathered successfully.")
    return profile, forecast, budget, debts_info, acc_sum, debt_det, sav_det

# Test data gathering for user_0001
target_user = 'user_0001'
profile, forecast, budget, debts_info, acc_sum, debt_det, sav_det = gather_user_context(target_user)

print(f"\nProfile: {profile.get('profile_type')}")
print(f"Total Accounts Balance: {acc_sum.get('total_balance'):,.2f} EGP")
print(f"Total Debt Obligations: {debt_det.get('monthly_obligations'):,.2f} EGP/month")


Gathering data for user_0001...
✅ Data gathered successfully.

Profile: growing
Total Accounts Balance: 44,212.36 EGP
Total Debt Obligations: 5,817.62 EGP/month


## 2. Knowledge Base (RAG Engine)
We use **ChromaDB** and `sentence-transformers` to store and retrieve financial guidelines. This ensures the LLM's advice is based on our curated financial framework rather than generic internet knowledge.


In [4]:
from utils.rag_engine import build_knowledge_base, get_relevant_context

# 1. Build the ChromaDB knowledge base from Markdown files in the /knowledge folder
print("Building Knowledge Base...")
build_knowledge_base()
print("✅ Knowledge Base ready.")

# 2. Test the retrieval system
test_query = "What is the 50/30/20 rule?"
print(f"\nTest Query: '{test_query}'")
retrieved_context = get_relevant_context(test_query, n_results=2)

print("\nRetrieved Context Snippet:")
print("-" * 50)
print(retrieved_context[:500] + "...\n")
print("-" * 50)


Building Knowledge Base...
✅ Knowledge Base ready.

Test Query: 'What is the 50/30/20 rule?'


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


Retrieved Context Snippet:
--------------------------------------------------
## Relevant Financial Knowledge


**Source**: budgeting_basics.md (relevance: 0.22)

# Budgeting Basics for Freelancers

## The 50/30/20 Rule
The 50/30/20 rule is a simple budgeting framework:
- **50% Needs**: Essential expenses like rent, utilities, food, transportation, and medical costs.
- **30% Wants**: Non-essential spending like entertainment, dining out, subscriptions, and shopping.
- **20% Savings & Debt**: Emergency fund, savings goals, and debt repayment.

For freelancers with irregula...

--------------------------------------------------


In [5]:
import subprocess
import sys

# Ensure required packages are installed
packages = ['chromadb', 'sentence-transformers', 'google-generativeai']
for package in packages:
    try:
        __import__(package.replace('-', '_'))
        print(f"✅ {package} is installed")
    except ImportError:
        print(f"Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])
        print(f"✅ {package} installed successfully")

✅ chromadb is installed
✅ sentence-transformers is installed
Installing google-generativeai...
✅ google-generativeai installed successfully


## 3. Prompt Engineering
We construct a strict `SYSTEM_PROMPT` to control the AI's persona, and a `USER_CONTEXT_TEMPLATE` to inject the user's financial data clearly.


In [6]:
from utils.prompts import SYSTEM_PROMPT, build_user_context, format_chat_prompt

# Build the structured string representing the user's financial state
user_context_string = build_user_context(
    profile, forecast, budget, debts_info,
    account_summary=acc_sum,
    debt_detail=debt_det,
    savings_detail=sav_det,
)

print("Generated User Context (Snippet):")
print("=" * 60)
# Print just the first 15 lines to avoid cluttering the notebook
print("\n".join(user_context_string.split("\n")[:15]))
print("...\n" + "=" * 60)


Generated User Context (Snippet):

## User Financial Profile
- **Profile Type**: growing
- **Average Monthly Income**: 5,964.34
- **Income Stability Score**: 74.9/100
- **Platform Diversity**: 0.97
- **Primary Skills**: N/A

## Income Overview
- **Predicted Monthly Income (next 3 months)**: 8,574.68
- **Income Trend**: 📉 Decreasing
- **Model Used**: global_lstm

## Account Summary
- **Total Balance (all accounts)**: 44,212.36
...


## 4. Specialized Analytical Functions
Instead of relying purely on the LLM to do math, we provide hard-coded financial logic tools that the system or the UI can use to give definitive answers.


In [7]:
from features.financial_advisor import can_i_afford, installment_impact, savings_plan, which_debt_first

print("--- Affordability Check ---")
afford = can_i_afford(target_user, 15000)
print(f"Target Purchase: 15,000 EGP")
print(f"Verdict: {afford['verdict']}")
print(f"Method: {afford['method']}")

print("\n--- Installment Impact Analysis ---")
impact = installment_impact(target_user, monthly_amount=500, months=12)
print(f"Installment: 500 EGP/month")
print(f"Affordable: {impact['affordable']}")
print(f"Risk Level: {impact['risk_level']}")
if impact['warnings']:
    for w in impact['warnings']:
        print(f"  [!] {w}")

print("\n--- Savings Plan Generator ---")
plan = savings_plan(target_user, "Apartment Down Payment", 200000, 36)
print(f"Goal: {plan['goal_name']}")
print(f"Verdict: {plan['verdict']}")
print(f"Recommendation: {plan['recommendation']}")


--- Affordability Check ---
Target Purchase: 15,000 EGP
Verdict: affordable_from_savings
Method: You can afford this from your available balance (after emergency fund).

--- Installment Impact Analysis ---
Installment: 500 EGP/month
Affordable: False
Risk Level: high
  [!] Monthly surplus (-7,836) should be >= 1.5× installment (750)
  [!] New debt-to-income ratio (71.7%) exceeds 35% threshold
  [!] You won't be able to maintain 10% savings after this installment
  [!] You already have 3 active debts

--- Savings Plan Generator ---
Goal: Apartment Down Payment
Verdict: not_feasible
Recommendation: You currently have a deficit. Focus on increasing income or reducing expenses first.


## 5. The LLM Chatbot Integration
We combine the User Context, the RAG Context, and the user's message, then send it to the **Gemini 1.5 Flash** API (optimized for free tier with reliable quotas).

In [8]:
from features.financial_advisor import chat

# Only run if API key is present
if os.environ.get("GEMINI_API_KEY"):
    question = "بناءً على ميزانيتي، هل أقدر أتحمل قسط جديد بـ 500 جنيه شهرياً؟"
    print(f"User: {question}\n")
    
    print("Generating AI Response... (This may take a few seconds)")
    response = chat(target_user, question)
    
    print("\n🤖 Advisor:")
    print("=" * 60)
    print(response["response"])
    print("=" * 60)
    
    print(f"\nSources Used: {response['sources']}")
    print(f"User Context Used: {response['user_context_used']}")
else:
    print("Cannot run chat simulation without GEMINI_API_KEY.")


User: بناءً على ميزانيتي، هل أقدر أتحمل قسط جديد بـ 500 جنيه شهرياً؟

Generating AI Response... (This may take a few seconds)

🤖 Advisor:
أهلاً بك! دعنا نلقي نظرة دقيقة على وضعك المالي لنرى ما إذا كان بإمكانك تحمل قسط جديد بقيمة 500 جنيه شهرياً.

بناءً على أرقامك الحالية، للأسف الشديد **لا أنصحك بتحمل قسط جديد بقيمة 500 جنيه شهرياً في الوقت الحالي.**

دعني أوضح لك لماذا، بناءً على بياناتك:

1.  **عجز شهري كبير**: متوسط دخلك الشهري هو 5,964.34 جنيه مصري، بينما متوسط مصروفاتك الشهرية 7,303.32 جنيه مصري، بالإضافة إلى التزامات ديون شهرية بقيمة 5,817.62 جنيه مصري.
    *   إجمالي مصاريفك والتزاماتك الشهرية = 7,303.32 (مصروفات) + 5,817.62 (أقساط) = 13,120.94 جنيه مصري.
    *   هذا يعني أن لديك **عجزاً شهرياً حقيقياً يصل إلى 7,156.6 جنيه مصري** (5,964.34 - 13,120.94).
    *   مع هذا العجز، إضافة قسط جديد سيزيد الضغط المالي بشكل كبير.

2.  **نسبة الدين للدخل مرتفعة جداً**: التزامات ديونك الشهرية الحالية (5,817.62 جنيه) تمثل حوالي 97.5% من متوسط دخلك الشهري (5,964.34 جنيه). وهذا يتجاوز بكثير الن